# 工具调用

## 1. 简单调用


In [12]:
from typing import Literal

from langchain_core import tools
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain_core.messages import HumanMessage,SystemMessage,AnyMessage
from pydantic import BaseModel
from rich import print as rprint


load_dotenv()
url = os.getenv("BASE_URL")
key = os.getenv("API_KEY")
model = os.getenv("DEEPSEEK_MODEL")


# 创建模型
model = init_chat_model(
    base_url = url,
    api_key = key,
    model=model,
    model_provider="deepseek"
)

# 声明工具
@tool
def get_weather(city_name:str) -> str:
    """获取指定城市的天气""" # 工具通过python doc方式进行描述的指定
    print(f"{city_name} 被工具调用执行")
    return f"{city_name} : 今日天气有雨"


# 绑定工具后返回一个新的对象
model_bind_tool = model.bind_tools([get_weather]) # 传入工具列表

# 创建消息
messages : list[AnyMessage] = [
    SystemMessage("你是一只猫娘"),
    HumanMessage("调用工具搜索告诉我今天北京天气怎么样？")
]

resp = model_bind_tool.invoke(messages)
print("=" * 20 + "工具调用返回" + "=" * 20 +'\n')
resp.pretty_print()
# rprint(resp)
# print("=" * 40)
messages.append(resp)

for tool_call in resp.tool_calls:
    if tool_call.get("name") == "get_weather":
        tool_resp = get_weather.invoke(tool_call)
        print("=" * 20 + "工具结果" + "=" * 20 + '\n')
        print(type(tool_resp))
        messages.append(tool_resp)

print("=====================> messages <=====================")
for msg in messages:
    msg.pretty_print()
print("=====================> messages <=====================")
resp = model_bind_tool.invoke(messages)

resp.pretty_print()


====================工具调用返回====================

================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_WxuAsHACaJlgrzAaegw83907)
 Call ID: call_00_WxuAsHACaJlgrzAaegw83907
  Args:
    city_name: 北京
北京 被工具调用执行
====================工具结果====================

<class 'langchain_core.messages.tool.ToolMessage'>
=====================> messages <=====================
================================ System Message ================================

你是一只猫娘
================================ Human Message =================================

调用工具搜索告诉我今天北京天气怎么样？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_WxuAsHACaJlgrzAaegw83907)
 Call ID: call_00_WxuAsHACaJlgrzAaegw83907
  Args:
    city_name: 北京
================================= Tool Message =================================
Name: get_weather

北京 : 今日天气有雨
=====================> messages <=====================
=========

## 2. 使用函数作为工具

### 2.1 了解convert_to_openai_tool

执行`model.bind_tools([get_weather])`，底层最终会调用`convert_to_openai_tool`生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

### 2.2 description说明

注意，首行默认为工具描述，参数描述需要与工具描述之间隔一个空行。

`Args`、`Returns` 等为固定写法。其中`Args` 添加了参数的描述之后，该参数必须要执行类型，否则报错


In [17]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str = '北京'):
    """
    查询城市的天气

    Args:
        city: 城市名称

    Returns:
        返回的城市天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '城市名称', 'type': 'string'}},
            'type': 'object'
        }
    }
}

## 3. 使用 `@Tool` 装饰器声明工具

- `description` ：描述。在同时声明了description和docstring的情况下，description的优先级更高
- `name_or_callable` ：更改工具名称


In [19]:
@tool(description="获取具体城市的天气情况",name_or_callable="getWeather",args_schema=WeatherInput)
def get_weather(city : str):
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取具体城市的天气情况',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 3.1. 自定义args_schema

- 当 args_schema 和 函数入参同时定义时，args_schema 优先级更高
- Literal：可以使用 Literal类型限定参数为固定选项。
    - Literal ：表示字段不能是任意某种类型的值，而只能是几个固定字面量之一。

In [21]:
from pydantic import BaseModel,Field
from typing import Literal

class WeatherInput(BaseModel):
    city : str = Field(
        description="城市名称",
        default="北京"
    )
    unit: Literal["hello","world"]


@tool(description="获取具体城市的天气情况",name_or_callable="getWeather",args_schema=WeatherInput)
def get_weather(city : str,unit:str):
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))




{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '获取具体城市的天气情况',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '城市名称', 'type': 'string'},
                'unit': {'enum': ['hello', 'world'], 'type': 'string'}
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}

### 3.2. 使用Json Schema定义

在 LangChain 中，还可以直接使用 JSON Schema 字典 来定义工具的参数模式。这种方式提供了极大的灵活性。

因为工具参数模式可以基于数据库配置或用户输入在 运行时动态生成 ，所以这种方式特别适合参数结构需要动态生成的场景

In [22]:

json_schema = {
    'properties': {
        'city': {'default': '北京', 'description': '具体的城市111', 'type': 'string'},
        'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
        'include_forecast': {
            'default': False,
            'description': '是否包含未来五天的天气预报111',
            'type': 'boolean'
        }
    },
    'required': ['unit'],
    'type': 'object'
}


@tool(args_schema=json_schema)
def get_weather(city : str,unit : str ="celsius",include_forecast : bool = True):
    """
    获取城市的天气
    """
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取城市的天气',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '具体的城市111', 'type': 'string'},
                'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'},
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五天的天气预报111',
                    'type': 'boolean'
                }
            },
            'required': ['unit'],
            'type': 'object'
        }
    }
}